[![Open In Colab](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module8/02-git.ipynb)](https://colab.research.google.com/github/datafieldtxt/datafieldnotebooks/blob/main/module8/02-git.ipynb)

# Module 8 Lesson 2 — Git for Python Projects

**Module 8: Best Practices & Real-World Python** | Estimated time: 35 minutes

## Learning Objectives

By the end of this lesson you will be able to:

- Configure Git and create a repository with `git init`
- Stage and commit changes, and read `git log` and `git diff`
- Write a proper `.gitignore` file for Python projects
- Work with branches: create, switch, merge, and resolve conflicts
- Use `git stash` to shelve work-in-progress
- Understand GitHub workflows: push, pull, fork, and pull requests
- Apply **conventional commits** and **semantic versioning**

In [ ]:
import os
import subprocess

# Configure git identity for this Colab session
!git config --global user.name "PyPath Learner"
!git config --global user.email "learner@pypath.dev"
!git config --global init.defaultBranch main

# Create a fresh project directory for our demo
os.makedirs("/tmp/myproject", exist_ok=True)
os.chdir("/tmp/myproject")

print("Working directory:", os.getcwd())
print("Git version:", subprocess.getoutput("git --version"))

## Initialising a Repository

Every Git repository starts with `git init`. This creates a hidden `.git/` folder that tracks all history.

In [ ]:
# Initialise the repo
!git init
!git status

In [ ]:
%%writefile README.md
# My Python Project

A demo project for learning Git best practices.

## Installation

```bash
pip install -r requirements.txt
```

## Usage

```python
from myapp import greet
print(greet("World"))
```

In [ ]:
%%writefile myapp.py
def greet(name: str) -> str:
    """Return a personalised greeting."""
    return f"Hello, {name}!"


if __name__ == "__main__":
    print(greet("World"))

## .gitignore — Keeping Junk Out of the Repo

A `.gitignore` file tells Git which files and directories to never track. For Python projects there are several important patterns.

In [ ]:
%%writefile .gitignore
# Python bytecode
__pycache__/
*.py[cod]
*$py.class
*.pyc

# Virtual environments
venv/
.venv/
env/
.env/

# Environment variables / secrets
.env
.env.local
*.env

# Distribution / packaging
dist/
build/
*.egg-info/
*.egg

# Testing
.pytest_cache/
.coverage
htmlcov/

# IDEs
.vscode/
.idea/
*.swp
*.swo

# OS files
.DS_Store
Thumbs.db

# Jupyter
.ipynb_checkpoints/

# mypy
.mypy_cache/

In [ ]:
# Stage files and make the first commit
!git add README.md myapp.py .gitignore
!git status
print()
!git commit -m "feat: initial project scaffold with greet function"

## Core Git Commands — add, commit, log, diff

Let's make a second change so we can see `git diff` and `git log` in action.

In [ ]:
%%writefile myapp.py
def greet(name: str) -> str:
    """Return a personalised greeting."""
    return f"Hello, {name}!"


def farewell(name: str) -> str:
    """Return a farewell message."""
    return f"Goodbye, {name}. See you soon!"


if __name__ == "__main__":
    print(greet("World"))
    print(farewell("World"))

In [ ]:
# See unstaged changes
print("=== git diff (unstaged) ===")
!git diff myapp.py

# Stage and see staged changes
!git add myapp.py
print("\n=== git diff --staged ===")
!git diff --staged myapp.py

In [ ]:
!git commit -m "feat: add farewell function"

print("\n=== git log ===")
!git log --oneline --graph --all

print("\n=== git log (verbose) ===")
!git log --format="%H | %an | %ad | %s" --date=short

## Branching — Isolated Lines of Development

Branches let you develop features in isolation without affecting the main codebase.

```
main:    A --- B
                \
feature:         C --- D
```

In [ ]:
# Create and switch to a feature branch
!git branch feature/add-utils
!git checkout feature/add-utils

print("=== Current branches ===")
!git branch -v

In [ ]:
%%writefile utils.py
from typing import Sequence


def average(numbers: Sequence[float]) -> float:
    """Return the arithmetic mean of a sequence of numbers."""
    if not numbers:
        raise ValueError("Cannot calculate average of empty sequence")
    return sum(numbers) / len(numbers)


def clamp(value: float, minimum: float, maximum: float) -> float:
    """Clamp value to [minimum, maximum] range."""
    return max(minimum, min(maximum, value))

In [ ]:
!git add utils.py
!git commit -m "feat: add average and clamp utility functions"

# Switch back to main and merge
!git checkout main
!git merge feature/add-utils --no-ff -m "Merge feature/add-utils into main"

print("\n=== Log after merge ===")
!git log --oneline --graph --all

## Merge Conflicts — Detecting and Resolving

A merge conflict happens when two branches change the same part of the same file. Git cannot auto-merge and asks you to choose.

In [ ]:
# Create a conflict: two branches edit the same line
!git checkout -b branch-a

# Branch A changes the greeting
with open("myapp.py", "r") as f:
    content = f.read()
content_a = content.replace('return f"Hello, {name}!"', 'return f"Greetings, {name}! [from branch-a]"')
with open("myapp.py", "w") as f:
    f.write(content_a)

!git add myapp.py
!git commit -m "style: change greeting message in branch-a"

# Go back to main and create branch-b
!git checkout main
!git checkout -b branch-b

content_b = content.replace('return f"Hello, {name}!"', 'return f"Welcome, {name}! [from branch-b]"')
with open("myapp.py", "w") as f:
    f.write(content_b)

!git add myapp.py
!git commit -m "style: change greeting message in branch-b"

# Try to merge branch-a into branch-b — CONFLICT
print("=== Attempting merge (expect conflict) ===")
!git merge branch-a || echo "CONFLICT detected — manual resolution needed"

In [ ]:
# See conflict markers
with open("myapp.py") as f:
    print(f.read())

In [ ]:
# Resolve by choosing a final version
resolved = '''def greet(name: str) -> str:
    """Return a personalised greeting."""
    return f"Hello, {name}! (resolved)"  # final agreed version


def farewell(name: str) -> str:
    """Return a farewell message."""
    return f"Goodbye, {name}. See you soon!"


if __name__ == "__main__":
    print(greet("World"))
    print(farewell("World"))
'''

with open("myapp.py", "w") as f:
    f.write(resolved)

!git add myapp.py
!git commit -m "fix: resolve merge conflict in greet function"

print("\n=== Conflict resolved ===")
!git log --oneline --graph --all | head -10

## git stash — Shelving Work-in-Progress

`git stash` temporarily saves uncommitted changes so you can switch branches without committing incomplete work.

In [ ]:
!git checkout main

# Start editing a file but don't finish
with open("myapp.py", "a") as f:
    f.write("\n# TODO: add more functions (work in progress)\n")

print("=== Status with WIP changes ===")
!git status

# Stash the work
!git stash push -m "WIP: adding more functions"

print("\n=== Status after stash ===")
!git status

print("\n=== Stash list ===")
!git stash list

# Re-apply the stash
!git stash pop
print("\n=== Status after stash pop ===")
!git status

## GitHub Workflow — Push, Pull, Fork

The typical GitHub workflow for open-source collaboration:

In [ ]:
github_workflow = """
# ── INITIAL SETUP ────────────────────────────────────────────────────────────

# 1. Create a new repo on GitHub (via UI), then:
git remote add origin https://github.com/your-username/myproject.git
git push -u origin main          # -u sets upstream tracking

# ── DAILY WORKFLOW ───────────────────────────────────────────────────────────

git pull origin main             # fetch + merge remote changes
# ... make changes ...
git add -p                       # interactive staging (patch by patch)
git commit -m "feat: your change"
git push

# ── FEATURE BRANCH WORKFLOW ──────────────────────────────────────────────────

git checkout -b feature/my-feature
# ... develop ...
git push -u origin feature/my-feature
# Open Pull Request on GitHub UI
# After PR is merged:
git checkout main
git pull origin main
git branch -d feature/my-feature  # clean up local branch

# ── FORK WORKFLOW (contributing to someone else's project) ───────────────────

# 1. Fork the repo on GitHub UI
# 2. Clone YOUR fork
git clone https://github.com/your-username/upstream-project.git
cd upstream-project

# 3. Add the original repo as 'upstream'
git remote add upstream https://github.com/original-author/upstream-project.git

# 4. Keep your fork in sync
git fetch upstream
git checkout main
git merge upstream/main
"""

print(github_workflow)

## Conventional Commits & Semantic Versioning

### Conventional Commits format
```
<type>[optional scope]: <description>

[optional body]

[optional footer(s)]
```

| Type | When to use |
|---|---|
| `feat` | A new feature |
| `fix` | A bug fix |
| `docs` | Documentation only |
| `style` | Formatting (no logic change) |
| `refactor` | Code change that is neither fix nor feat |
| `test` | Adding or fixing tests |
| `chore` | Build process, dependency updates |
| `perf` | A performance improvement |

### Semantic Versioning: `MAJOR.MINOR.PATCH`
- **PATCH** (`1.0.1`): backwards-compatible bug fix
- **MINOR** (`1.1.0`): backwards-compatible new feature
- **MAJOR** (`2.0.0`): breaking change

In [ ]:
# Examples of good vs bad commit messages
good_commits = [
    "feat(auth): add JWT token refresh endpoint",
    "fix(api): handle None response from payment gateway",
    "docs: update README with Docker instructions",
    "test(utils): add parametrized tests for clamp function",
    "chore: upgrade requests to 2.31.0",
    "perf(db): add index on users.email column",
    "feat!: drop support for Python 3.9 (BREAKING CHANGE)",
]

bad_commits = [
    "fix stuff",
    "WIP",
    "asdfgh",
    "changes",
    "update code",
    "it works now",
]

print("GOOD commit messages:")
for c in good_commits:
    print(f"  ✓  {c}")

print("\nBAD commit messages (avoid these):")
for c in bad_commits:
    print(f"  ✗  {c}")

## Practice Exercises

**Exercise 1 — Full Git workflow**
In a new directory called `/tmp/git-practice`:
1. Run `git init`
2. Create a `.gitignore` for Python
3. Write a `calculator.py` with an `add(a, b)` function
4. Make an initial commit
5. Create a branch `feature/multiply`, add a `multiply(a, b)` function, commit, and merge back to `main`
6. Run `git log --oneline --graph` to see the history

**Exercise 2 — Simulate a merge conflict**
Creating two branches that edit the same line in `calculator.py`, merge them, resolve the conflict, and commit the resolution.

**Exercise 3 — Write conventional commits**
Rewrite these bad commit messages as conventional commits:
- `"fixed bug"` → (hint: what was the bug? make something up)
- `"added stuff to README"` →
- `"new version"` →
- `"speed improvement"` →